In [2]:
import torch, torch.nn as nn
import math, copy
print(torch.__version__)

2.11.0+cpu


In [3]:
#Cell G1: multiclass loss by hand
logits = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([0])
sm = torch.softmax(logits, dim=1)

print(sm.round(decimals=4), sm.sum())
manual = -torch.log(sm[0, 0])
print(manual)
print(nn.CrossEntropyLoss()(logits, target))

for t in (0, 1, 2):
  print(t, round(nn.CrossEntropyLoss()(logits, torch.tensor([t])).item(), 4))

tensor([[0.6590, 0.2424, 0.0986]]) tensor(1.0000)
tensor(0.4170)
tensor(0.4170)
0 0.417
1 1.417
2 2.317


- The loop changes the TRUE class from 0 to 2. Why does the loss grow from 0.417 to 2.317?
  * because the when the Class is 0, the model confidently predicted because the highest raw logit is 2.0 so the loss is smaller. But when Class is 2 was actually the correct answer, the model assigned to lowest raw logit 0.1, so the loss is bigger.
  The loss is -log of the probability on the true class. Class 2 only gets 0.0986 (its logit is the lowest), so -log(0.0986) = 2.317, much bigger than -log(0.6590) = 0.417 for class 0.

- nn.CrossEntropyLoss takes raw logits. So should the model's last layer include a Softmax? (Link it
to what you wrote for BCEWithLogitsLoss on Day 14.)
  * No, because CrossEntropyLoss applies softmax and the log inside, in a numerically stable way, just like BCEWithLogitsLoss applies the sigmoid inside on Day 14. So the model returns raw logits in both cases. Adding Softmax or Sigmoid in the model would apply it twice and give wrong, squashed probabilities.

In [7]:
#Cell G2: label shape and dtype, and the starting loss
logits = torch.zeros(5, 10)
target = torch.tensor([0, 1, 2, 3, 4])
loss = nn.CrossEntropyLoss()(logits, target)
print(target.dtype, target.shape)
print(round(loss.item(), 4), round(math.log(10), 4))

torch.int64 torch.Size([5])
2.3026 2.3026


- Churn labels were float with shape (N, 1). What dtype and shape must the labels have for
CrossEntropyLoss, and which tensor method or dtype name gives you that (hint: torch.long)?
  * For CrossEntropyLoss the dtype must be long and the shape must be (N, ).
  .to(torch.long) will give the dtype.

- Why should an UNTRAINED 10-class model start with a loss near 2.3026? What would it mean if your
first-epoch loss were 50?
  * An untrained model gives all 10 classes about the same probability, 1/10, so the loss is -log(0.1) = 2.3026 = ln(10). A first-epoch loss of 50 would mean the model is confidently wrong, with huge logits, so I would check that the inputs are scaled, the learning rate is sane and the loss is a mean.

In [8]:
#Cell G3: predictions and accuracy by hand
lg = torch.tensor([[2.0, 1.0, 0.1],
                   [0.1, 3.0, 0.2],
                   [1.0, 1.2, 1.1],
                   [0.0, 0.0, 5.0]])
y = torch.tensor([0, 1, 0, 2])
pred = lg.argmax(dim=1)
print(pred)
print((pred == y).float().mean())

tensor([0, 1, 1, 2])
tensor(0.7500)


- which sample is wrong, what did the model predict, and what was the true class?
  * sample 3 (third row)[1.0, 1.2, 1.1] is wrong because the highest logit is 1.2 and the model predicted 1 but the answer is 0. Just because it guessed Class 1 by a tiny margin. The model predicted 3 out of 4 correctly, it gives 0.75 accuracy.

In [17]:
#Cell G4: what is inside state_dict
torch.manual_seed(0)
tiny = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))
for k, v in tiny.state_dict().items():
  print(k, tuple(v.shape))
print(len(tiny.state_dict()), sum(p.numel() for p in tiny.parameters()))

0.weight (3, 4)
0.bias (3,)
2.weight (2, 3)
2.bias (2,)
4 23


Count the 23 by hand: (4 x 3 + 3) + (3 x 2 + 2) = 23

- why is there no entry for layer "1" (the ReLU)?
  * The keys are named after each layer’s position in the Sequential. Layer 1 is the ReLU, a fixed function with no parameters, so state_dict only lists layers 0 and 2: 0.weight, 0.bias, 2.weight, 2.bias.

In [18]:
#Cell G5: save, load, and prove it worked
torch.save(tiny.state_dict(), 'tiny.pt')
torch.manual_seed(1)
new = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))
x = torch.ones(1, 4)
with torch.no_grad():
  print(tiny(x), new(x))
new.load_state_dict(torch.load('tiny.pt'))
with torch.no_grad():
  print(new(x))
  print(torch.equal(tiny(x), new(x)))

tensor([[ 0.4794, -0.1188]]) tensor([[-0.3676,  0.0545]])
tensor([[ 0.4794, -0.1188]])
True


- Why do we save state_dict() and not the whole model object? (What does the file contain, and what must you supply yourself to load it?)
  * The file holds only names and weight tensors, no code, so it is small and doesn’t break when my code or PyTorch changes. To load it I must write the same architecture myself (same layers and shapes) and then call load_state_dict.

- What did the error message in your try tell you?
  * The error tells that the shape does not match. For 0.weight (3,4) and the current model is (5,4), for 0.bias (3) and the current model is (5) and lastly 2.weight (2,3) and the current model is (2,5)

In [13]:
#Cell G6: the trap: state_dict is a reference, not a copy
sd = tiny.state_dict()
first = sd["0.weight"].clone()
with torch.no_grad():
  tiny[0].weight.add_(1.0)
print(torch.equal(sd["0.weight"], first))


sd2 = copy.deepcopy(tiny.state_dict())
first2 = sd2["0.weight"].clone()
with torch.no_grad():
  tiny[0].weight.add_(1.0)
print(torch.equal(sd2["0.weight"], first2))


False
True


- In a training loop that wants to remember the BEST epoch, what goes wrong if you store model.state_dict() without a copy?
  * state_dict() returns references to the live weights, not copies. If I store it at the best epoch and keep training, the stored ‘best’ keeps changing with the model. At the end I would save the last epoch’s weights while believing they were the best, and nothing would raise an error.

In [15]:
#Cell G7: why model.eval() matters (a preview of Dropout on Day 17)
torch.manual_seed(0)
drop = nn.Dropout(0.5)
x = torch.ones(1, 8)
drop.train()
print(drop(x))
drop.eval()
print(drop(x))

tensor([[0., 0., 2., 0., 0., 0., 2., 2.]])
tensor([[1., 1., 1., 1., 1., 1., 1., 1.]])


- Dropout switches off random units in train mode and scales the survivors by 1/(1-0.5) = 2. Why does eval mode switch it off?
  * Because eval mode switches dropout off because during validation and testing, you want your model to make consistent, deterministic, and highly stable predictions. Leaving dropout active would mean randomly blinding half of your neurons on every forward pass, which introduces artificial noise and causes your final evaluation metrics. In train mode the survivors are multiplied by 2, so the average output stays the same as with Dropout off. So in eval mode, Dropout just passes the values through unchanged (all 1.0 in the cell).

- Our Day 14-16 networks have no Dropout. Why is it still correct to call model.eval() before validation?
  * Our networks have no Dropout or BatchNorm, so eval() changes nothing today. I still call it because tomorrow I add Dropout, and a validation loop that forgets eval() would silently give noisy, wrong numbers. Writing it every time means I never have to remember.
